# Step 2 · How people commute: mode shares from the INSEE census

**Question:** what share of the working population of each city cycles to work, every year from 2006 to 2022?

The French census (INSEE, table **NAV2A**) counts, for each commune, the employed people by the *main mode of
transport used to go to work*. This notebook reads the 17 yearly tables once and computes, for every studied city,
the share (%) of each mode: no transport, walking, two-wheelers, car, public transport.

**Important:** before 2017 INSEE groups bicycles and motorbikes together ("two-wheelers"). From 2017 they are
separate. The model of step 3 therefore uses **`Deux-roues total`** (bicycle + motorbike) so that the series is
consistent; the bicycle-only share (`Vélo`) exists only from 2017.

| | |
|---|---|
| **Input** | `data/insee_nav2a_extract/NAV2A_<year>.csv` (in git: the rows of the studied communes) — or the full raw tables `data/insee_raw/NAV2A_<year>/*.xls(x)` if you downloaded them (see [`data/README.md`](../data/README.md)) |
| **Output** | `data/insee_modal_shares/<city>_<INSEE code>_modal_shares_2006_2022.csv` (in git) |
| **Run time** | 2 seconds from the extract; about 7 minutes from the raw Excel files |

The extract contains the complete INSEE rows of the 12 communes, so both sources give exactly the same result.
Code: [`cyclelts/insee.py`](../cyclelts/insee.py).

In [1]:
# Setup: find the repository root and make the `cyclelts` package importable.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cyclelts").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from cyclelts import config, insee

CITIES = config.CITIES + config.EXTRA_CITIES     # 11 studied cities + Nancy
YEARS = range(2006, 2023)
RECOMPUTE = True        # False = only load the CSV files already in data/insee_modal_shares/
USE_RAW_IF_AVAILABLE = True   # the full raw tables are used when all years are present in data/insee_raw/

## 1. Read the INSEE tables (raw files or the extract in the repository)

In [3]:
files = insee.find_nav2a_files(config.INSEE_RAW_DIR, list(YEARS))
if USE_RAW_IF_AVAILABLE and len(files) == len(YEARS):
    source = config.INSEE_RAW_DIR          # full national tables (~600 MB, downloaded by you)
else:
    source = config.INSEE_EXTRACT_DIR      # rows of the studied communes (in git)
print("Reading the NAV2A tables from", source.relative_to(ROOT))

if RECOMPUTE:
    tables = insee.modal_shares_for_communes([c.insee for c in CITIES], source, YEARS)
    shares = {}
    for c in CITIES:
        path = insee.modal_share_file(config.INSEE_SHARES_DIR, c.slug, c.insee)
        path.parent.mkdir(parents=True, exist_ok=True)
        tables[c.insee].to_csv(path, index=False, encoding="utf-8-sig")
        shares[c.name] = tables[c.insee]
    print("Saved one CSV per city in", config.INSEE_SHARES_DIR.relative_to(ROOT))
else:
    shares = {c.name: insee.read_modal_shares(config.INSEE_SHARES_DIR, c.slug) for c in CITIES}

display(shares["Grenoble"].round(2))

Reading the NAV2A tables from data\insee_raw


2006: read BTX_TD_NAV2A_2006.xls


2007: read BTX_TD_NAV2A_2007.xls


2008: read BTX_TD_NAV2A_2008.xls


2009: read BTX_TD_NAV2A_2009.xls


2010: read BTX_TD_NAV2A_2010.xls


... (9 outputs left out)

2020: read TD_NAV2A_2020.xlsx


2021: read TD_NAV2A_2021.xlsx


2022: read TD_NAV2A_2022.xlsx
Saved one CSV per city in data\insee_modal_shares


,Year,CODGEO,Commune,Pas de déplacement,Marche,Deux-roues total,Vélo,Deux-roues motorisé,Voiture,TC
0,2006,38185,,2.46,14.54,9.56,NaN,NaN,48.37,25.07
1,2007,38185,,2.77,14.52,10.37,NaN,NaN,46.86,25.48
2,2008,38185,,2.79,14.45,10.62,NaN,NaN,45.49,26.64
3,2009,38185,,2.82,14.27,10.91,NaN,NaN,44.50,27.49
4,2010,38185,,2.81,14.65,11.27,NaN,NaN,43.20,28.07
5,2011,38185,Grenoble,2.85,14.26,12.21,NaN,NaN,42.50,28.18
6,2012,38185,Grenoble,2.75,14.31,12.79,NaN,NaN,41.58,28.58
7,2013,38185,Grenoble,2.68,14.42,13.80,NaN,NaN,40.87,28.23
8,2014,38185,Grenoble,2.70,14.32,15.00,NaN,NaN,39.91,28.07
9,2015,38185,Grenoble,2.65,13.85,15.69,NaN,NaN,39.73,28.08


## 2. Two-wheeler share of every city

The vertical dotted line marks 2017, when INSEE starts separating bicycles from motorbikes.

In [4]:
fig = go.Figure()
for name, table in shares.items():
    fig.add_scatter(x=table.Year, y=table["Deux-roues total"], mode="lines+markers", name=name)
fig.add_vline(x=2017, line_dash="dot", annotation_text="bike / motorbike separated")
fig.update_layout(template="plotly_white", height=500, title="Commuting share of two-wheelers (INSEE)",
                  xaxis_title="Year", yaxis_title="Two-wheelers (bike + motorbike), % of workers")
fig.show()

latest = pd.DataFrame({name: t.set_index("Year").loc[2022, ["Deux-roues total", "Vélo", "Voiture", "TC", "Marche"]]
                       for name, t in shares.items()}).T.sort_values("Deux-roues total", ascending=False)
latest.columns = [insee.MODE_LABELS_EN[c] for c in latest.columns]
print("2022 commuting mode shares (%):")
display(latest.round(1))

2022 commuting mode shares (%):


,Two-wheelers (bike + motorbike),Bicycle,Car,Public transport,Walking
Grenoble,23.5,22.7,32.3,27.9,13.0
Bordeaux,21.5,19.0,33.9,27.9,11.5
Strasbourg,21.2,20.6,37.7,27.5,9.9
Rennes,15.9,14.8,40.7,28.0,11.8
Nantes,15.8,14.1,42.0,28.2,10.0
Lyon,14.3,12.8,29.2,37.9,14.7
Paris,13.6,10.1,9.0,60.7,11.7
Angers,12.8,11.4,53.8,17.0,12.6
Chambéry,11.5,10.3,56.6,13.6,14.6
Annecy,11.1,9.6,63.9,11.3,9.2


## 3. All modes for one city

Stars mark the years where the two-wheeler share **decreased** compared with the previous year.

In [5]:
CITY = "Grenoble"
t = shares[CITY].sort_values("Year").reset_index(drop=True)
change = t["Deux-roues total"].diff()
fig = go.Figure()
for col in ["Voiture", "TC", "Deux-roues total", "Marche", "Pas de déplacement"]:
    fig.add_scatter(x=t.Year, y=t[col], mode="lines+markers", name=insee.MODE_LABELS_EN[col])
down = change < 0
fig.add_scatter(x=t.Year[down], y=t["Deux-roues total"][down], mode="markers+text", name="two-wheelers decrease",
                marker=dict(symbol="star", size=14, line=dict(width=1, color="black")),
                text=[f"{v:.2f} pp" for v in change[down]], textposition="top center")
fig.add_vline(x=2017, line_dash="dot")
fig.update_layout(template="plotly_white", height=500, title=f"{CITY} — commuting mode shares (INSEE)",
                  xaxis_title="Year", yaxis_title="% of workers")
fig.show()
display(t.loc[down, ["Year", "Deux-roues total"]].assign(change_pp=change[down]).round(3))

,Year,Deux-roues total,change_pp
13,2019,18.169,-0.037
